# Addressing reviewer R4C6

To strengthen the case, the authors could consider including additional slices—particularly those with the worst DSC—to show how the model behaves in failure cases.

Let's first find the bottom 10% dice scores from the baseline model:

In [13]:
RUNS = {
    "amos": {
        "baseline": "../bundles/amos22_baseline_dice_ce_nl",
    },
    "acdc": {
        "baseline": "../bundles/acdc17_baseline_dice_ce_1",
    },
    "kits": {
        "baseline": "../bundles/kits23_baseline_dice_ce_nl",
    },
    "brats": {
        "baseline": "../bundles/brats21_baseline_dice_ce_nl",
    },
}

In [14]:
import os
import pandas as pd
import numpy as np

SEED = 12345

In [15]:
def get_lowest_10_percent_cases(run_path, seed=SEED):
    """
    Load the mean_dice_raw.csv file and return the bottom 10 scores across all classes
    Returns a dataframe with filename, class, and dice score for each low-scoring case
    """
    csv_path = os.path.join(run_path, f'seed_{seed}', "inference_results", "mean_dice_raw.csv")
    
    if not os.path.exists(csv_path):
        print(f"Warning: {csv_path} does not exist")
        return None
    
    df = pd.read_csv(csv_path)
    
    # Get all class columns (exclude 'filename' and 'mean')
    class_cols = [col for col in df.columns if col.startswith('class')]
    
    # Create a list to store all (filename, class, score) tuples
    all_scores = []
    
    for _, row in df.iterrows():
        filename = row['filename']
        for class_col in class_cols:
            score = row[class_col]
            if pd.notna(score):  # Only include non-NaN scores
                all_scores.append({
                    'filename': filename,
                    'class': class_col,
                    'dice_score': score
                })
    
    # Convert to dataframe and sort by dice score
    scores_df = pd.DataFrame(all_scores)
    scores_df_sorted = scores_df.sort_values('dice_score', ascending=True)
    
    # Get bottom 10
    bottom_10 = scores_df_sorted.head(10)
    
    return bottom_10

## Find lowest 10% Dice scores for each dataset

In [16]:
# Analyze each dataset
for dataset_name, runs in RUNS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'='*60}\n")
    
    baseline_path = runs["baseline"]
    bottom_cases = get_lowest_10_percent_cases(baseline_path)
    
    if bottom_cases is not None:
        print(f"Lowest 10 Dice scores across all classes:\n")
        print(bottom_cases.to_string(index=False))
        print(f"\nDice score range: {bottom_cases['dice_score'].min():.4f} - {bottom_cases['dice_score'].max():.4f}")
        print(f"Unique cases affected: {bottom_cases['filename'].nunique()}")
    else:
        print(f"Could not find data for {dataset_name}")
    print()


Dataset: AMOS

Lowest 10 Dice scores across all classes:

        filename   class  dice_score
amos_0308.nii.gz class14    0.000000
amos_0287.nii.gz class13    0.013801
amos_0293.nii.gz  class3    0.013992
amos_0108.nii.gz  class3    0.031044
amos_0203.nii.gz  class3    0.040467
amos_0356.nii.gz class10    0.111588
amos_0356.nii.gz class12    0.116810
amos_0106.nii.gz  class3    0.119816
amos_0056.nii.gz  class4    0.136072
amos_0208.nii.gz  class3    0.153846

Dice score range: 0.0000 - 0.1538
Unique cases affected: 9


Dataset: ACDC

Lowest 10 Dice scores across all classes:

                 filename  class  dice_score
patient103_frame11.nii.gz class0    0.344422
patient142_frame12.nii.gz class0    0.516523
patient142_frame12.nii.gz class2    0.626247
patient142_frame01.nii.gz class1    0.676941
patient114_frame11.nii.gz class0    0.686046
patient129_frame01.nii.gz class1    0.707405
patient105_frame10.nii.gz class2    0.725167
patient134_frame15.nii.gz class2    0.743185
patient10

## Optional: Save results to dictionary for later use

In [17]:
# Store results in a dictionary
failure_cases = {}

for dataset_name, runs in RUNS.items():
    baseline_path = runs["baseline"]
    bottom_cases = get_lowest_10_percent_cases(baseline_path)
    
    if bottom_cases is not None:
        failure_cases[dataset_name] = bottom_cases

# Display summary
print("Summary of failure cases found:")
for dataset, cases_df in failure_cases.items():
    print(f"\n{dataset}: {len(cases_df)} lowest scores")
    print(f"  Unique cases: {cases_df['filename'].nunique()}")
    print(f"  Dice range: {cases_df['dice_score'].min():.4f} - {cases_df['dice_score'].max():.4f}")
    print(f"  Classes affected: {', '.join(sorted(cases_df['class'].unique()))}")

Summary of failure cases found:

amos: 10 lowest scores
  Unique cases: 9
  Dice range: 0.0000 - 0.1538
  Classes affected: class10, class12, class13, class14, class3, class4

acdc: 10 lowest scores
  Unique cases: 9
  Dice range: 0.3444 - 0.7504
  Classes affected: class0, class1, class2

kits: 10 lowest scores
  Unique cases: 8
  Dice range: 0.0000 - 0.3047
  Classes affected: class0, class1

brats: 10 lowest scores
  Unique cases: 9
  Dice range: 0.0000 - 0.3977
  Classes affected: class0, class1, class2


In [18]:
bottom_10_cases = [
"results_plots/seg_plots/seg_plots_amos_all/amos_0308_prostate or uterus.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0287_bladder.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0293_gallbladder.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0108_gallbladder.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0203_gallbladder.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0356_right_adrenal_gland.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0356_duodenum.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0106_gallbladder.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0056_esophagus.pdf",
"results_plots/seg_plots/seg_plots_amos_all/amos_0208_gallbladder.pdf",

"results_plots/seg_plots/seg_plots_acdc_all/patient103_frame01_right_ventricle.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient142_frame01_right_ventricle.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient142_frame01_left_ventricle.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient142_frame01_myocardium.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient114_frame01_right_ventricle.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient129_frame01_myocardium.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient105_frame10_left_ventricle.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient134_frame15_left_ventricle.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient108_frame09_left_ventricle.pdf",
"results_plots/seg_plots/seg_plots_acdc_all/patient129_frame08_myocardium.pdf",

"results_plots/seg_plots/seg_plots_kits_hec/case_00020_tumour.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00463_tumour.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00020_kidney_mass.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00121_tumour.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00463_kidney_mass.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00466_tumour.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00207_tumour.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00001_tumour.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00264_tumour.pdf",
"results_plots/seg_plots/seg_plots_kits_hec/case_00189_kidney_mass.pdf",

"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_01176_Enhancing Tumour (ET).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_01530_Tumour Core (TC).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_01530_Enhancing Tumour (ET).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_01512_Enhancing Tumour (ET).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_01480_Tumour Core (TC).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_00331_Tumour Core (TC).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_00493_Whole Tumour (WT).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_01364_Whole Tumour (WT).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_01490_Tumour Core (TC).pdf",
"results_plots/seg_plots/seg_plots_brats_hec/BraTS2021_00656_Tumour Core (TC).pdf",
]

## Copy the bottom 10 cases to a new directory

In [19]:
import shutil

# Create the destination directory
dest_dir = "../results_plots/revisions_lowest_10_dsc_cases"
os.makedirs(dest_dir, exist_ok=True)

# Copy each file to the destination directory
for file_path in bottom_10_cases:
    # Construct full path from workspace root
    source_path = os.path.join("..", file_path)
    
    # Get the filename from the path
    filename = os.path.basename(file_path)
    
    # Construct destination path
    dest_path = os.path.join(dest_dir, filename)
    
    # Copy the file
    if os.path.exists(source_path):
        shutil.copy2(source_path, dest_path)
        print(f"Copied: {filename}")
    else:
        print(f"Warning: File not found - {source_path}")

print(f"\nAll files copied to: {dest_dir}")

Copied: amos_0308_prostate or uterus.pdf
Copied: amos_0287_bladder.pdf
Copied: amos_0293_gallbladder.pdf
Copied: amos_0108_gallbladder.pdf
Copied: amos_0203_gallbladder.pdf
Copied: amos_0356_right_adrenal_gland.pdf
Copied: amos_0356_duodenum.pdf
Copied: amos_0106_gallbladder.pdf
Copied: amos_0056_esophagus.pdf
Copied: amos_0208_gallbladder.pdf
Copied: patient103_frame01_right_ventricle.pdf
Copied: patient142_frame01_right_ventricle.pdf
Copied: patient142_frame01_left_ventricle.pdf
Copied: patient142_frame01_myocardium.pdf
Copied: patient114_frame01_right_ventricle.pdf
Copied: patient129_frame01_myocardium.pdf
Copied: patient105_frame10_left_ventricle.pdf
Copied: patient134_frame15_left_ventricle.pdf
Copied: patient108_frame09_left_ventricle.pdf
Copied: patient129_frame08_myocardium.pdf
Copied: case_00020_tumour.pdf
Copied: case_00463_tumour.pdf
Copied: case_00020_kidney_mass.pdf
Copied: case_00121_tumour.pdf
Copied: case_00463_kidney_mass.pdf
Copied: case_00466_tumour.pdf
Copied: case_